# 05 · Señales de compra y venta

**Objetivo:** convertir los indicadores del notebook 04 en **señales de compra y de venta** con reglas explícitas: régimen de mercado, filtro de liquidez y tres mecanismos de entrada.

> Las señales son **técnicas e informativas**, no una recomendación de inversión.

**Entradas:**
- `data/cache/precios_historicos.parquet` (notebook 02): precios diarios ajustados, 10 años, 1.000 empresas
- `data/cache/indicadores.parquet` (notebook 04): los indicadores por empresa y día

**Salidas** (la leen los notebooks 06 y 09 y la app):
- `data/cache/senales.parquet`: una fila por empresa y día, con el régimen, la liquidez, cada mecanismo y la señal final

In [1]:
import time

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

from funciones import (
    ADX_UMBRAL_TENDENCIA,
    CARPETA_CACHE,
    SESIONES_CALENTAMIENTO,
    detectar_senales,
)

load_dotenv();

## Glosario: ¿qué mide cada indicador?

| Indicador | En una línea | Uso aquí |
|---|---|---|
| **RSI** | Mide de 0 a 100 si el precio ha subido o bajado demasiado rápido (< 30 sobrevendido, > 70 sobrecomprado) | Señal en régimen lateral |
| **ADX** | Mide la **fuerza** de la tendencia, sin dirección (≥ 25 = hay tendencia) | Decide el régimen |
| **CMF** | Flujo de dinero de −1 a +1: si el volumen acompaña a las subidas o a las bajadas | Confirma el cruce de EMA |
| **EMA** | Media móvil exponencial: da más peso a los días recientes. Se cruzan la de 4 y la de 8 sesiones | Señal en régimen de tendencia |
| **Donchian** | Máximo y mínimo de las 20 sesiones anteriores (sin hoy) | Señal de ruptura |

**Conclusión:** el ADX decide el régimen; el cruce de EMA (confirmado por la CMF), el RSI y Donchian dan las señales.

## 1. ¿De qué datos partimos?

Se unen los precios y los indicadores por fecha y empresa.

In [2]:
precios = pd.read_parquet(CARPETA_CACHE / "precios_historicos.parquet")
indicadores = pd.read_parquet(CARPETA_CACHE / "indicadores.parquet")
datos = precios.merge(indicadores, on=["fecha", "ticker"], how="inner", validate="one_to_one")
datos = datos.sort_values(["ticker", "fecha"]).reset_index(drop=True)

# Número de sesión dentro del histórico de cada empresa (0 = su primer día)
datos["numero_sesion"] = datos.groupby("ticker").cumcount()

print("Filas:", len(datos), "· Empresas:", datos["ticker"].nunique())
print("Fechas:", datos["fecha"].min().date(), "→", datos["fecha"].max().date())

Filas: 2281091 · Empresas: 1000
Fechas: 2016-10-10 → 2026-10-09


**Conclusión:** 1.000 empresas con unos 10 años de datos diarios cada una.

## 2. ¿Se puede uno fiar del ADX desde el primer día?

Cada día se clasifica con el ADX: **tendencia** si ADX ≥ 25 y **lateral** si es menor. El problema es que el ADX sale inflado en las primeras sesiones de cada empresa, hasta que acumula historia. El gráfico compara el % de empresas en tendencia cada mes, **sin** descartar nada y **descartando las primeras 60 sesiones** de cada empresa (calentamiento).

In [3]:
datos["en_tendencia"] = datos["adx"] >= ADX_UMBRAL_TENDENCIA
con_adx = datos[datos["adx"].notna()]
pasado_calentamiento = con_adx[con_adx["numero_sesion"] >= SESIONES_CALENTAMIENTO]

sin_calentamiento = con_adx.groupby(pd.Grouper(key="fecha", freq="ME"))["en_tendencia"].mean() * 100
con_calentamiento = pasado_calentamiento.groupby(pd.Grouper(key="fecha", freq="ME"))["en_tendencia"].mean() * 100

grafico = go.Figure()
grafico.add_trace(go.Scatter(x=sin_calentamiento.index, y=sin_calentamiento, name="sin calentamiento"))
grafico.add_trace(go.Scatter(x=con_calentamiento.index, y=con_calentamiento, name="con calentamiento de 60 sesiones"))
grafico.update_layout(title="% de empresas en régimen de tendencia (ADX ≥ 25), por mes",
                      xaxis_title="Mes", yaxis_title="% de empresas-día en tendencia")
grafico.show(renderer="plotly_mimetype")

print(f"Con calentamiento: {pasado_calentamiento['en_tendencia'].mean():.0%} del tiempo en tendencia")

Con calentamiento: 38% del tiempo en tendencia


Sin calentamiento, octubre y noviembre de 2016 (el arranque del histórico) salen con más del 80 % de las empresas en tendencia, casi como el desplome de la COVID en marzo de 2020 (86 %), sin que pasara nada parecido en el mercado. Es un artefacto del cálculo: con el calentamiento, enero de 2017 baja al 33 % y el pico más alto es **marzo de 2020**.

**Conclusión:** las primeras **60 sesiones** de cada empresa no tienen régimen ni señales.

## 3. ¿Qué días son demasiado poco líquidos para operar?

Algunas empresas apenas se negociaban en EEUU al principio de su histórico. Una señal en esos días no se podría ejecutar. Se mide el **volumen en dólares** (cierre × volumen), con la mediana de las últimas 20 sesiones, y se mira cuánto quedaría fuera con distintos mínimos:

In [4]:
datos["volumen_dolares"] = datos["close"] * datos["volume"]
datos["volumen_dolares_mediano"] = datos.groupby("ticker")["volumen_dolares"].transform(
    lambda serie: serie.rolling(20).median())

filas = []
for minimo in [100_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000]:
    por_debajo = datos["volumen_dolares_mediano"] < minimo
    filas.append({
        "mínimo (USD/día)": minimo,
        "% de filas por debajo": round(100 * por_debajo.mean(), 2),
        "empresas afectadas en algún momento": datos.loc[por_debajo, "ticker"].nunique(),
    })
pd.DataFrame(filas)

,mínimo (USD/día),% de filas por debajo,empresas afectadas en algún momento
0,100000,0.75,39
1,1000000,1.60,85
2,2000000,2.18,103
3,5000000,3.32,159
4,10000000,5.27,245


**Conclusión:** mínimo de **1 M USD/día**. Quita solo el 1,6 % de los días, las etapas claramente inoperables, sin tocar el periodo normal de casi ninguna empresa. Los precios no se modifican: el filtro solo decide si un día puede tener señal.

## 4. ¿Cuándo hay señal de compra o de venta?

Hay tres mecanismos independientes. La venta es siempre el espejo de la compra.

| Mecanismo | Régimen | Compra | Venta |
|---|---|---|---|
| **1. Cruce de EMA** | Solo tendencia (ADX ≥ 25) | EMA(4) cruza por encima de EMA(8) y CMF ≥ +0,07 | EMA(4) cruza por debajo de EMA(8) y CMF ≤ −0,07 |
| **2. Reversión por RSI** | Solo lateral (ADX < 25) | El RSI vuelve a subir por encima de 30 | El RSI vuelve a bajar por debajo de 70 |
| **3. Ruptura de Donchian** | Cualquiera | El cierre supera el máximo de las 20 sesiones anteriores | El cierre cae bajo el mínimo de las 20 sesiones anteriores |

Hay señal si se cumple **cualquiera** de los tres, el día es líquido y ya pasó el calentamiento. Si un día tuviera compra y venta a la vez se marcaría como **conflicto**, sin operación.

Las reglas están en `detectar_senales` (`funciones.py`). Ejemplo con las últimas señales de Apple:

In [5]:
columnas_indicadores = list(indicadores.columns.drop(["fecha", "ticker"]))

precios_apple = datos[datos["ticker"] == "AAPL"].set_index("fecha")
senales_apple = detectar_senales(precios_apple[["close", "volume"]], precios_apple[columnas_indicadores])
senales_apple[senales_apple["senal"].notna()].tail(8)

,regimen,liquido,compra_ema,venta_ema,compra_rsi,venta_rsi,compra_donchian,venta_donchian,senal
fecha,,,,,,,,,
2026-07-16,tendencia,True,False,False,False,False,True,False,compra
2026-07-27,tendencia,True,False,False,False,False,True,False,compra
2026-07-28,tendencia,True,False,False,False,False,True,False,compra
2026-09-01,lateral,True,False,False,False,False,True,False,compra
2026-09-11,lateral,True,False,False,False,False,True,False,compra
2026-09-17,lateral,True,False,False,False,False,True,False,compra
2026-09-21,lateral,True,False,False,False,False,True,False,compra
2026-09-22,lateral,True,False,False,False,False,True,False,compra


**Conclusión:** cada fila dice qué mecanismo se activó, en qué régimen estaba el día y si era líquido.

## 5. ¿Cuántas señales salen y en qué años?

Se aplica la misma función a cada una de las 1.000 empresas.

In [6]:
inicio_calculo = time.time()
tablas = []
for ticker, datos_ticker in datos.groupby("ticker"):
    datos_ticker = datos_ticker.set_index("fecha")
    senales_ticker = detectar_senales(datos_ticker[["close", "volume"]], datos_ticker[columnas_indicadores])
    senales_ticker.insert(0, "ticker", ticker)
    tablas.append(senales_ticker.reset_index())

senales = pd.concat(tablas, ignore_index=True)
print(f"Calculado en {time.time() - inicio_calculo:.1f} s · filas: {len(senales)}")

Calculado en 2.1 s · filas: 2281091


Días en que se activa cada mecanismo, sin y con el filtro de liquidez. Un mismo día puede activar varios a la vez.

In [7]:
mecanismos = ["compra_ema", "compra_rsi", "compra_donchian", "venta_ema", "venta_rsi", "venta_donchian"]
recuento = pd.DataFrame({
    "días (sin filtro)": senales[mecanismos].sum(),
    "días (con filtro de liquidez)": senales.loc[senales["liquido"], mecanismos].sum(),
})
recuento

,días (sin filtro),días (con filtro de liquidez)
compra_ema,10525,10307
compra_rsi,4888,4759
compra_donchian,175272,173369
venta_ema,8836,8550
venta_rsi,8525,8431
venta_donchian,114842,113159


Señales finales de compra y de venta, en total y por año:

In [8]:
dias_con_senal = senales["senal"].isin(["compra", "venta"])
print(senales["senal"].value_counts().to_string())
print(f"Días con señal: {dias_con_senal.mean():.1%} de todos los empresa-día")

por_anio = senales[dias_con_senal].groupby([senales["fecha"].dt.year, "senal"]).size()
por_anio = por_anio.rename("días").reset_index().rename(columns={"fecha": "año"})
grafico = px.bar(por_anio, x="año", y="días", color="senal", barmode="group",
                 title="Días con señal por año y sentido")
grafico.show(renderer="plotly_mimetype")

senal
compra    187837
venta     129549
Días con señal: 13.9% de todos los empresa-día


- **Donchian domina** porque en una subida sostenida el cierre marca un máximo varios días seguidos. Solo el 64 % son rupturas nuevas; el notebook 09 cuenta solo el primer día de cada racha.
- Ningún día tiene conflicto.
- **2022 y 2018**, los dos años bajistas, son los que más ventas tienen en proporción a las compras.

**Conclusión:** las reglas reaccionan al mercado como se espera. Qué pasa con la acción después de cada señal, y si cambia según las noticias, se mide en el notebook 09.

## 6. ¿Cómo se ve sobre un gráfico? (Apple, último año)

Triángulos verdes = compra, rojos = venta. Fondo gris = régimen de tendencia.

In [9]:
desde = precios_apple.index.max() - pd.DateOffset(years=1)
p = precios_apple[precios_apple.index >= desde]
s = senales_apple[senales_apple.index >= desde]

grafico = go.Figure()
grafico.add_trace(go.Candlestick(x=p.index, open=p["open"], high=p["high"], low=p["low"], close=p["close"], name="Precio"))

compras = s.index[s["senal"] == "compra"]
ventas = s.index[s["senal"] == "venta"]
grafico.add_trace(go.Scatter(x=compras, y=p.loc[compras, "low"], mode="markers", name="compra",
                             marker={"symbol": "triangle-up", "size": 11, "color": "green"}))
grafico.add_trace(go.Scatter(x=ventas, y=p.loc[ventas, "high"], mode="markers", name="venta",
                             marker={"symbol": "triangle-down", "size": 11, "color": "red"}))

# Fondo gris en los tramos de tendencia: se recorre día a día y se anota dónde empieza y acaba cada tramo
inicio_tramo = None
dia_anterior = None
for dia, regimen in s["regimen"].items():
    if regimen == "tendencia" and inicio_tramo is None:
        inicio_tramo = dia
    if regimen != "tendencia" and inicio_tramo is not None:
        grafico.add_vrect(x0=inicio_tramo, x1=dia_anterior, fillcolor="gray", opacity=0.12, line_width=0)
        inicio_tramo = None
    dia_anterior = dia
if inicio_tramo is not None:
    grafico.add_vrect(x0=inicio_tramo, x1=dia_anterior, fillcolor="gray", opacity=0.12, line_width=0)

grafico.update_layout(title="AAPL: señales del último año (gris = tendencia)", height=550, xaxis_rangeslider_visible=False)
grafico.show(renderer="plotly_mimetype")

**Conclusión:** las compras aparecen sobre todo en subidas (rupturas) y en rebotes tras caídas en régimen lateral (RSI).

## 7. ¿Qué se guarda?

In [10]:
ruta_senales = CARPETA_CACHE / "senales.parquet"
senales.to_parquet(ruta_senales, index=False)
print(f"Guardado: {ruta_senales.name} - {len(senales)} filas - {ruta_senales.stat().st_size / 1e6:.1f} MB")

Guardado: senales.parquet - 2281091 filas - 2.2 MB


**Conclusión:** `senales.parquet` guardado con 2.281.091 filas (2,2 MB), una por empresa y día.

## Comprobaciones

In [11]:
# Una fila de señales por cada fila de datos, en el mismo orden
assert senales[["fecha", "ticker"]].equals(datos[["fecha", "ticker"]])

# Ninguna señal durante el calentamiento ni en días no líquidos
calentamiento = datos["numero_sesion"] < SESIONES_CALENTAMIENTO
assert not senales.loc[calentamiento, mecanismos].any().any()
assert senales.loc[~senales["liquido"], "senal"].isna().all()

# Cada mecanismo respeta su régimen: EMA solo en tendencia, RSI solo en lateral
assert (senales.loc[senales["compra_ema"] | senales["venta_ema"], "regimen"] == "tendencia").all()
assert (senales.loc[senales["compra_rsi"] | senales["venta_rsi"], "regimen"] == "lateral").all()

# Solo valores de señal previstos
assert set(senales["senal"].dropna().unique()) <= {"compra", "venta", "conflicto"}
print("Señales: OK")

Señales: OK
